In [17]:
from MagicDB import MagicDB as mDB
from DatabaseAPI.BinanceDB import BinanceDB as bDB
from datetime import datetime as dt
import pytz
import numpy as np
from torch.nn import Linear, ReLU, Softmax, Sequential, CrossEntropyLoss
import torch
import torch.nn as nn
def get_best_sells(timeseries, window):
    '''
    Gets indices of the best times to sell
    '''
    sells = []
    for i in range(0,len(timeseries)-window, window):
        targets = list(timeseries[i:i+window])
        sells.append(i + targets.index(max(targets)))
    return sells

def get_best_buys(timeseries, window):
    buys = []
    for i in range(0,len(timeseries)-window, window):
        targets = list(timeseries[i:i+window])
        buys.append(i + targets.index(min(targets)))
    return buys

def generate_Y(timeseries, window, graph = False):
    '''
    Given a pandas df of price data, generate y data where each 
    point y_i is a vector where index 0 is sell, index 1 is hold
    and index 2 is buy
    '''
    buys = get_best_buys(timeseries, window)
    sells = get_best_sells(timeseries, window)
    if graph:
        plt.plot(timeseries, marker = 6, 
                 label ='Buys', markevery =buys)
        plt.plot(timeseries, marker = 7, 
                 label = 'Sells', markevery =sells)
        plt.show()
    final = []
    for i in range(len(timeseries)):
        if i in buys:
            final.append([0,0,1])
        elif i in sells:
            final.append([1,0,0])
        else:
            final.append([0,1,0])
    return np.array(final)
            
def get_profit(strategy, ts):
    '''
    Given strategy that gets best times to buy, find the profit
    made by adding each sell price to each buy price
    '''
    assert strategy.shape[0] == ts.shape[0]
    price_weights = np.dot(ts, np.array([1,0,-1]).reshape(1,-1))
    result = np.sum(np.multiply(strategy, price_weights))
    return result    

In [2]:
#####NEURAL NET TIME BABY##########

In [3]:
#Input: MAVGs(10,20,30,40,50) (scaled from 0-1)
#, Volume moving avg(10,20,30,40)(scaled from 0-1)
#Output: Softmax
#Loss Function: NLLM

In [4]:
#Xi = [MA10,MA20,MA30,MA40,MA50, V10,V20,V30,V40]

In [6]:
def generate_X(data):
    '''
    Given dataframe with "Open" and "Volume" fields, create a 
    10xn size numpy array with n observations
    '''
    prices = np.array(data['Open'])
    volumes = np.array(data['Volume'])
    X = []
    for i in range(50, len(prices)):            
        MA10 = sum(prices[i-10:i])/10
        MA20 = sum(prices[i-20:i])/20
        MA30 = sum(prices[i-30:i])/30
        MA40 = sum(prices[i-40:i])/40
        MA50 = sum(prices[i-50:i])/50
        mavgs = [MA10,MA20,MA30,MA40,MA50]
        mini = min(mavgs)
        maxi = max(mavgs)
        rng = maxi-mini
        vec = []
        for j in mavgs:
            val = (j-mini)/rng
            vec.append(val)
        V10 = sum(volumes[i-10:i])/10
        V20 = sum(volumes[i-20:i])/20
        V30 = sum(volumes[i-30:i])/30
        V40 = sum(volumes[i-40:i])/40
        V50 = sum(volumes[i-50:i])/50
        vavgs = [V10,V20,V30,V40,V50]
        mini = min(vavgs)
        maxi = max(vavgs)
        rng = maxi-mini
        for j in vavgs:
            val = (j-mini)/rng
            vec.append(val)
        X.append(vec)
    X = np.array(X).T #X DONE
    return X
def generate_dataset(dataframe, window):
    '''
    Given a pandas dataframe, and the window to get the best buys
    and sells over, return X,Y
    '''
    X = generate_X(dataframe)
    Y = generate_Y(dataframe["Open"][50:], window)
    return X,Y

In [18]:
start = dt(2020,5,4)
end = dt(2020,5,5)
m = mDB('1m', dt(2020,5,4), dt(2020,5,5), bDB())
m.track_historical_crypto_ticker('ETHBTC', pytz.utc.localize(start),
                                 pytz.utc.localize(end))
X,Y = generate_dataset(m.timeseries["ETHBTC"], 50)

In [19]:
X

array([[1.        , 1.        , 1.        , ..., 0.39909639, 0.30449391,
        0.22255193],
       [0.41841415, 0.4163486 , 0.40897984, ..., 0.35109187, 0.43049139,
        0.56750742],
       [0.4648318 , 0.45610687, 0.4318876 , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.62681065, 0.63525915, 0.66379035, ..., 1.        , 1.        ,
        1.        ],
       [0.62623187, 0.64924614, 0.69929984, ..., 0.23762185, 0.37998753,
        0.5668452 ],
       [0.44530138, 0.41410034, 0.37591952, ..., 0.        , 0.18487483,
        0.46065784]])

In [20]:
Y

array([[0, 1, 0],
       [0, 1, 0],
       [0, 1, 0],
       ...,
       [0, 1, 0],
       [0, 1, 0],
       [0, 1, 0]])

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()